In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
from langchain_core.tools import tool


In [ ]:
from langchain_tavily import TavilySearch
tavily_search = TavilySearch(
    max_results=2,
    topic="general",
    include_answer=True
)
@tool
def web_search(query:str) -> str:

    """
    Uses tavily tool to search the web for up-to-date information and returns relevant results with sources.
    """

    return tavily_search.invoke({"query":query})['answer']
# web_search("what is recent ai news")

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b"
)


In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

# Files are saved in langgraph state

# By default we provide a StateBackend
agent = create_deep_agent(model=llm)

# Under the hood, it looks like
agent = create_deep_agent(
    model=llm,
    backend=StateBackend(),
)

In [ ]:
result=agent.invoke({
    "messages":[{
        "role":"user","content":(
            "Create a file at /notes/todo.txt with exactly the below content:\n"
            "1. Record video\n"
            "2. Edit video\n"
            "3.Upload video\n"
            "Then tell me you have done it."
        )
    }]
})

In [ ]:
result

In [ ]:
files=result.get('files',{})

if files:
    print(f"{len(files)}")
    for path,content in files.items():
        print(f"{path}\n{'-'*40}\n{content}")
else :
    print("No files found in state. Either the agent did not write a file, or the backend isn't wired up correctly")

In [ ]:
followup=agent.invoke({
    "messages":result["messages"]+[{
        "role":"user",
        "content":"Read /notes/todo.txt back to me."
    }],
    "files":result.get("files",{})
    # pass the cirtual filesystem along
})

followup

In [ ]:
print(followup["messages"][-1].content)

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
ROOT="."
agent = create_deep_agent(
    model=llm,
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

In [ ]:
result=agent.invoke({
    "messages":[{
        "role":"user","content":(
            "Create a file at /notes/todo.txt with exactly the below content:\n"
            "1. Record video\n"
            "2. Edit video\n"
            "3.Upload video\n"
            "Then tell me you have done it."
        )
    }]
})

In [ ]:
result

In [ ]:
from pathlib import Path
# disk_path=Path(ROOT)
disk_path=Path("./notes/todo.txt")

if disk_path.exists():
    print("File exists on disk")
    print(disk_path.resolve())
    print(disk_path.read_text())
else:
    print(f"expected file not at {disk_path.resolve()}")
    print("The agent may ot have called the write tool, or the path mapping differs")


In [ ]:
followup= agent.invoke({
    "messages":result["messages"]+[{
        "role":"user",
        "content":"Read /notes/todo.txt back to me verbatim."
    }]
    # pass the cirtual filesystem along
})

In [ ]:
followup

In [ ]:
followup["messages"][-1].content

### StoreBackend allows to create file in one thread and read from other threads

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

agent = create_deep_agent(
    model=llm,
    backend=StoreBackend(
        namespace=lambda rt: ("demo-user"),
    ),
    store=InMemoryStore(),  # Good for local dev; omit for LangSmith Deployment
)

In [ ]:
agent

In [ ]:
import uuid
thread_id={"configurable":{"thread_id":str(uuid.uuid4())}}
result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":(
            "Create a file at /notes/todo.txt with exactly this content\n"
            "1. Record video, edit and upload video\n"
            "Then tell me you have done it"
        )
    }],
    
}, config=thread_id)

In [ ]:
result

In [ ]:
thread_id_2={"configurable":{"thread_id":str(uuid.uuid4())}}
followup=agent.invoke({
    "messages":[{
        "role":"user",
        "content":(
       "Read /notes/todo.txt back to me verbatim"
        )
    }],
    
}, config=thread_id_2)

In [ ]:
followup